# Treemün 2.0.0 

This notebook demonstrates the complete Treemün 2.0.0 workflow. Pinus stands must always enter the planning horizon through a pre-thinning equation with a valid `next_equation_id`. If a policy's thinning age is already lower than the stand's initial biological age, the initial rotation is fully harvested in period 1 and the next rotation starts at biological age 1 on the original pre-thinning equation.

The notebook also covers thinning diagnostics, carbon accounting, three economic modes, terminal conditions, mathematical-model timing, bi-objective and three-objective optimization, green-up event conflicts, knee-point identification, and spatial export.


## 1. Imports and project paths

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display
from scipy.spatial import Delaunay, QhullError

import geopandas as gpd
import pyomo.environ as pyo
import treemun_sim as tm

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "examples" / "forest_stands.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

STANDS_FILE = PROJECT_ROOT / "examples" / "forest_stands.csv"
GEOMETRY_FILE = PROJECT_ROOT / "examples" / "treemun_landscape.gpkg"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Treemün version:", tm.__version__)
print("Python executable:", sys.executable)
print("Project root:", PROJECT_ROOT)


## 2. Reproducible configuration

In [ ]:
HORIZON = 30
DISCOUNT_RATE = 0.08
RELATIVE_GAP = 0.01
GREENUP_WINDOW = 2

FALLBACK_THINNING_FRACTION = 0.30
MINIMUM_CURVE_RESIDUAL_FRACTION = 0.10

WEIGHTED_FRONT_POINTS = 7
EPSILON_FRONT_POINTS = 7

# Six requested epsilon combinations, always using all 105 stands.
# Payoff anchors and previously solved epsilon points are reused whenever they
# are mathematically proven to remain optimal, so fewer than six new MILP
# solves may be required.
TRI_CARBON_POINTS = 3
TRI_GREENUP_POINTS = 2

# Solver parallelism. Treemün will use no more than this number of threads and
# will also respect common scheduler allocations such as SLURM_CPUS_PER_TASK.
SOLVER_THREAD_CAP = 8
# Set a positive value to impose a solver time limit; None leaves it unlimited.
SOLVER_TIME_LIMIT_SECONDS = None

# Terminal alternatives used in the sensitivity exercise.
MINIMUM_ENDING_STOCK_FRACTION = 0.50


PREFERRED_SOLVER = "cplex"
SOLVER_CANDIDATES = (
    "cplex",
    "appsi_highs",
    "highs",
    "cbc",
    "cplex_direct",
    "cplex_persistent",
    "appsi_cplex",
    "glpk",
)
# Exact counts are useful only for the bundled 105-stand example.
# Leave False when replacing the input landscape with another dataset.
VALIDATE_BUNDLED_EXAMPLE = False


## 3. Solver detection

In [ ]:
def solver_availability(candidates=SOLVER_CANDIDATES) -> dict[str, bool]:
    """Return Pyomo availability for each supported solver interface."""
    availability = {}
    for candidate in candidates:
        try:
            solver = pyo.SolverFactory(candidate)
            availability[candidate] = bool(
                solver is not None
                and solver.available(exception_flag=False)
            )
        except Exception:
            availability[candidate] = False
    return availability


def select_available_solver(preferred: str | None = None):
    """Select the preferred solver or the first available candidate."""
    availability = solver_availability()
    if preferred is not None:
        if preferred not in availability:
            raise ValueError(f"Unsupported solver name: {preferred}")
        if not availability[preferred]:
            raise RuntimeError(f"Requested solver {preferred!r} is unavailable.")
        return preferred, availability
    for candidate in SOLVER_CANDIDATES:
        if availability[candidate]:
            return candidate, availability
    raise RuntimeError("No supported MILP solver is available.")


def detect_solver_thread_budget(cap: int = 8) -> int:
    """Respect scheduler CPU allocations and avoid accidental oversubscription."""
    scheduler_variables = (
        "SLURM_CPUS_PER_TASK",
        "SLURM_CPUS_ON_NODE",
        "PBS_NP",
        "NSLOTS",
    )
    for variable in scheduler_variables:
        raw_value = os.getenv(variable)
        if raw_value:
            try:
                return max(1, min(int(cap), int(raw_value)))
            except ValueError:
                pass
    return max(1, min(int(cap), int(os.cpu_count() or 1)))


SOLVER_NAME, AVAILABLE_SOLVERS = select_available_solver(PREFERRED_SOLVER)
SOLVER_THREADS = detect_solver_thread_budget(SOLVER_THREAD_CAP)
SOLVER_OPTIONS = {
    "relative_gap": RELATIVE_GAP,
    "threads": SOLVER_THREADS,
    "time_limit_seconds": SOLVER_TIME_LIMIT_SECONDS,
    "tee": False,
}

print("Solver availability:")
for name, available in AVAILABLE_SOLVERS.items():
    print(f"  {name:20s}: {available}")
print("Selected solver:", SOLVER_NAME)
print("Solver threads requested:", SOLVER_THREADS)
print("Solver time limit (s):", SOLVER_TIME_LIMIT_SECONDS)


## 4. Inspect the lookup table

In [ ]:
lookup = tm.load_lookup_table()
print("Lookup rows:", len(lookup))
display(
    lookup.groupby(["species", "growth_curve"], dropna=False)
    .size()
    .rename("equations")
    .reset_index()
)

In [ ]:
pre_thinning = lookup.loc[
    lookup["growth_curve"].eq("pre_thinning_1250_700"),
    [
        "equation_id",
        "next_equation_id",
        "species",
        "zone",
        "site_index",
        "management_regime",
    ],
].copy()
post_curve_by_id = lookup.set_index("equation_id")["growth_curve"]
pre_thinning["next_growth_curve"] = pre_thinning["next_equation_id"].map(post_curve_by_id)
display(pre_thinning)
assert pre_thinning["next_equation_id"].notna().all()
assert pre_thinning["next_growth_curve"].eq("post_thinning_700_300").all()

## 5. Load all 105 stands and their geometries

In [ ]:
stands = tm.load_stand_table(STANDS_FILE)
landscape = gpd.read_file(GEOMETRY_FILE)
landscape["stand_id"] = landscape["stand_id"].astype(str)

print("Stands:", len(stands))
print("Total area (ha):", stands["area_ha"].sum())
print("Geometries:", len(landscape))
display(stands.groupby("species").size().rename("stands").reset_index())

assert set(stands["stand_id"].astype(str)) == set(landscape["stand_id"])
if VALIDATE_BUNDLED_EXAMPLE:
    assert len(stands) == 105


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
landscape.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.5)
ax.set_title("Treemün example landscape — 105 stands")
ax.set_axis_off()
plt.show()

## 6. Simulate every feasible stand-policy trajectory

The simulator validates that every Pinus input stand starts on a pre-thinning equation and reaches its post-thinning curve only through `next_equation_id`. Policies whose thinning age precedes the observed initial biological age trigger a complete period-1 harvest and rotation reset.


In [ ]:
start = time.perf_counter()
forest, policy_summary, ending_stock, harvested_by_period, carbon_by_period = tm.simulate_forest(
    stands_file=STANDS_FILE,
    horizon=HORIZON,
    include_carbon=True,
    return_carbon_for_optimization=True,
    fallback_thinning_fraction=FALLBACK_THINNING_FRACTION,
    minimum_curve_residual_fraction=MINIMUM_CURVE_RESIDUAL_FRACTION,
)
elapsed = time.perf_counter() - start

print("Input stands:", stands["stand_id"].nunique())
print("Simulated stands:", len({frame.iloc[0]["stand_id"] for frame in forest}))
print("Feasible stand-policy alternatives:", len(forest))
print("Annual trajectory rows:", sum(len(frame) for frame in forest))
print("Positive harvest coefficients:", len(harvested_by_period))
print(f"Simulation time: {elapsed:.2f} s")

assert len({frame.iloc[0]["stand_id"] for frame in forest}) == stands["stand_id"].nunique()
assert len(forest) >= stands["stand_id"].nunique()
if VALIDATE_BUNDLED_EXAMPLE:
    assert len(forest) == 780


In [ ]:
forest_long = tm.forest_to_long_table(forest)
policy_summary_table = pd.DataFrame(policy_summary)

display(
    forest_long.groupby("operation").size().rename("rows").reset_index()
)
display(policy_summary_table.head())

# Every stand-policy trajectory must begin from the same observed stock.
initial_states = forest_long.loc[forest_long["period"].eq(1)].copy()
initial_stock_consistency = (
    initial_states.groupby("stand_id")
    .agg(
        policies=("policy", "nunique"),
        distinct_initial_stocks=("standing_dry_wood_t_before_operation", "nunique"),
        initial_stock_t=("standing_dry_wood_t_before_operation", "first"),
    )
    .reset_index()
)
assert initial_stock_consistency["distinct_initial_stocks"].eq(1).all()

pinus_initial_states = initial_states.loc[initial_states["species"].eq(tm.PINUS)]
assert pinus_initial_states["growth_curve_before_operation"].str.startswith(
    "pre_thinning"
).all()

rotation_resets = forest_long.loc[
    forest_long["initial_rotation_reset_triggered"]
].copy()
print("Policies with a period-1 overdue-thinning rotation reset:", len(rotation_resets))
print("Pinus stands affected by at least one policy reset:", rotation_resets["stand_id"].nunique())
display(
    rotation_resets[
        [
            "stand_id",
            "policy",
            "stand_age",
            "thinning_age",
            "operation",
            "final_harvest_reason",
            "growth_curve_before_operation",
            "harvested_dry_wood_t",
        ]
    ].head(12)
)

# Confirm that period 2 restarts at age 1 on the pre-thinning curve.
if not rotation_resets.empty:
    reset_keys = rotation_resets[["stand_id", "policy"]].drop_duplicates()
    after_reset = forest_long.merge(reset_keys, on=["stand_id", "policy"], how="inner")
    after_reset = after_reset.loc[after_reset["period"].eq(2)]
    assert after_reset["stand_age"].eq(1).all()
    assert after_reset["growth_curve_before_operation"].str.startswith(
        "pre_thinning"
    ).all()


## 7. Diagnose the thinning calculation method

In [ ]:
thinning_rows = forest_long.loc[forest_long["operation"].eq("thinning")].copy()
thinning_diagnostics = (
    thinning_rows.groupby(
        ["thinning_calculation_method", "thinning_fallback_triggered"],
        dropna=False,
    )
    .agg(
        events=("stand_id", "size"),
        mean_candidate_residual_fraction=("candidate_curve_residual_fraction", "mean"),
        mean_applied_thinning_fraction=("applied_thinning_fraction", "mean"),
        minimum_remaining_stock_t=("standing_dry_wood_t_after_operation", "min"),
    )
    .reset_index()
)
display(thinning_diagnostics)
assert (thinning_rows["standing_dry_wood_t_after_operation"] > 0).all()

intensity_quantiles = (
    thinning_rows.groupby("thinning_calculation_method")["applied_thinning_fraction"]
    .quantile([0.00, 0.05, 0.25, 0.50, 0.75, 0.95, 1.00])
    .unstack()
    .rename(columns={0.00: "min", 0.05: "p05", 0.25: "p25", 0.50: "median", 0.75: "p75", 0.95: "p95", 1.00: "max"})
    .reset_index()
)
display(intensity_quantiles)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for method, group in thinning_rows.groupby("thinning_calculation_method"):
    ax.hist(
        group["applied_thinning_fraction"].dropna(),
        bins=np.linspace(0.0, 1.0, 21),
        alpha=0.55,
        label=method,
    )
ax.axvline(FALLBACK_THINNING_FRACTION, linestyle="--", linewidth=1.5, label="Fallback fraction")
ax.set_xlabel("Applied thinning fraction")
ax.set_ylabel("Number of thinning events")
ax.set_title("Distribution of thinning intensities — all feasible trajectories")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

regime_lookup = (
    lookup[["equation_id", "management_regime"]]
    .drop_duplicates(subset="equation_id")
    .rename(columns={"equation_id": "equation_id_before_operation"})
)
thinning_rows_with_regime = thinning_rows.merge(
    regime_lookup,
    on="equation_id_before_operation",
    how="left",
    validate="many_to_one",
)
assert thinning_rows_with_regime["management_regime"].notna().all()

by_regime = (
    thinning_rows_with_regime.groupby(
        ["management_regime", "thinning_calculation_method"],
        dropna=False,
    )
    .agg(
        events=("stand_id", "size"),
        mean_intensity=("applied_thinning_fraction", "mean"),
        median_intensity=("applied_thinning_fraction", "median"),
        maximum_intensity=("applied_thinning_fraction", "max"),
    )
    .reset_index()
)
display(by_regime)


In [ ]:
fallback_example = thinning_rows.loc[
    thinning_rows["thinning_calculation_method"].eq("fixed_fraction_fallback")
].iloc[0]
example_trajectory = next(
    frame
    for frame in forest
    if str(frame.iloc[0]["stand_id"]) == str(fallback_example["stand_id"])
    and str(frame.iloc[0]["policy"]) == str(fallback_example["policy"])
)
columns = [
    "period",
    "stand_age",
    "operation",
    "growth_curve_before_operation",
    "standing_dry_wood_t_before_operation",
    "candidate_post_thinning_dry_wood_t",
    "harvested_dry_wood_t",
    "standing_dry_wood_t_after_operation",
    "thinning_calculation_method",
    "post_thinning_continuity_adjustment_t",
]
display(example_trajectory.loc[:, columns])

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(
    example_trajectory["period"],
    example_trajectory["standing_dry_wood_t_before_operation"],
    marker="o",
    label="Before operation",
)
ax.plot(
    example_trajectory["period"],
    example_trajectory["standing_dry_wood_t_after_operation"],
    marker="s",
    label="After operation",
)
ax.set_xlabel("Period")
ax.set_ylabel("Dry wood (t)")
ax.set_title("Fallback thinning and continuous post-thinning growth")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 8. Carbon accounting

In [ ]:
trajectory_carbon = (
    forest_long.groupby(["stand_id", "policy", "species", "area_ha"], as_index=False)
    .agg(
        carbon_stock_time_tC_year=("carbon_stock_time_tC_year", "sum"),
        maximum_standing_carbon_tC=("standing_wood_carbon_tC_after_operation", "max"),
    )
)
trajectory_carbon["carbon_stock_time_tC_year_ha"] = (
    trajectory_carbon["carbon_stock_time_tC_year"] / trajectory_carbon["area_ha"]
)

candidate_carbon_summary = (
    trajectory_carbon.groupby("species", as_index=False)
    .agg(
        candidate_trajectories=("policy", "size"),
        mean_stock_time_tC_year_ha=("carbon_stock_time_tC_year_ha", "mean"),
        median_stock_time_tC_year_ha=("carbon_stock_time_tC_year_ha", "median"),
        maximum_standing_carbon_tC=("maximum_standing_carbon_tC", "max"),
    )
)
display(candidate_carbon_summary)
print(
    "This is a distribution of candidate trajectories, not a landscape total. "
    "A selected-plan carbon total is calculated after optimization."
)

sample = forest_long.iloc[0]
expected_carbon = (
    sample["standing_dry_wood_t_after_operation"]
    * sample["dry_wood_carbon_fraction"]
)
assert np.isclose(sample["standing_wood_carbon_tC_after_operation"], expected_carbon)

## 9. Define the three economic interpretations

In [ ]:
# ------------------------------------------------------------
# Chilean reference economic parameters
# Constant 2025 USD
# ------------------------------------------------------------

PRICE_BASE_YEAR = 2025
CURRENCY = "USD_2025"

# Real annual discount rate.
DISCOUNT_RATE = 0.08

GROSS_REVENUE_PARAMETERS = {
    "revenue_per_dry_t": {
        tm.PINUS: 44.07,
        tm.EUCALYPTUS: 43.66,
    }
}

NET_UNIT_VALUE_PARAMETERS = {
    "net_value_per_dry_t": {
        tm.PINUS: 15.70,
        tm.EUCALYPTUS: 24.59,
    }
}

BASE_DETAILED_CASH_FLOW_PARAMETERS = {
    "revenue_per_dry_t": {
        tm.PINUS: 44.07,
        tm.EUCALYPTUS: 43.66,
    },

    # No additional generic variable cost is added because harvesting
    # and transport costs are represented explicitly below.
    "variable_cost_per_dry_t": {
        tm.PINUS: 0.0,
        tm.EUCALYPTUS: 0.0,
    },

    "transport_cost_per_dry_t": {
        tm.PINUS: 15.76,
        tm.EUCALYPTUS: 10.59,
    },

    "thinning_cost_per_dry_t": {
        tm.PINUS: 11.03,
        tm.EUCALYPTUS: 0.0,
    },

    "final_harvest_cost_per_dry_t": {
        tm.PINUS: 12.61,
        tm.EUCALYPTUS: 8.47,
    },

    # Operational costs are represented per dry tonne, so these
    # per-hectare operation costs are set to zero to avoid duplication.
    "thinning_cost_per_ha": 0.0,
    "final_harvest_cost_per_ha": 0.0,

    "replanting_cost_per_ha": {
        tm.PINUS: 684.0,
        tm.EUCALYPTUS: 716.0,
    },

    "annual_management_cost_per_ha": 15.0,
    "replanting_delay_periods": 0,

    # No residual value in the basic comparison.
    "terminal_value_per_dry_t": 0.0,
}

TERMINAL_VALUE_PER_DRY_T = {
    tm.PINUS: 7.85,
    tm.EUCALYPTUS: 12.30,
}

# Main multiobjective analyses include a residual value for standing
# dry biomass remaining at the end of the planning horizon.
DETAILED_CASH_FLOW_PARAMETERS = {
    **BASE_DETAILED_CASH_FLOW_PARAMETERS,
    "terminal_value_per_dry_t": TERMINAL_VALUE_PER_DRY_T,
}

print("Economic parameters loaded.")
print("Currency:", CURRENCY)
print("Price base year:", PRICE_BASE_YEAR)
print("Real discount rate:", f"{DISCOUNT_RATE:.1%}")

In [ ]:
# ------------------------------------------------------------
# Economic evaluation under Chilean reference parameters
# Values are expressed in constant 2025 USD.
# ------------------------------------------------------------

economic_evaluations = {
    "gross_revenue": tm.evaluate_forest_economics(
        forest,
        economic_mode="gross_revenue",
        discount_rate=DISCOUNT_RATE,
        economic_parameters=GROSS_REVENUE_PARAMETERS,
    ),

    "net_unit_value": tm.evaluate_forest_economics(
        forest,
        economic_mode="net_unit_value",
        discount_rate=DISCOUNT_RATE,
        economic_parameters=NET_UNIT_VALUE_PARAMETERS,
    ),

    # The comparison uses the detailed model without terminal value,
    # so that the three accounting interpretations are easier to compare.
    "detailed_cash_flow": tm.evaluate_forest_economics(
        forest,
        economic_mode="detailed_cash_flow",
        discount_rate=DISCOUNT_RATE,
        economic_parameters=BASE_DETAILED_CASH_FLOW_PARAMETERS,
    ),
}


comparison = pd.concat(
    [
        evaluation.summary_by_policy.assign(
            economic_mode=mode,
            currency="USD_2025",
            discount_rate=DISCOUNT_RATE,
        )
        for mode, evaluation in economic_evaluations.items()
    ],
    ignore_index=True,
)


comparison_summary = (
    comparison.groupby(
        [
            "economic_mode",
            "economic_metric",
            "currency",
            "discount_rate",
        ],
        as_index=False,
    )
    .agg(
        minimum_economic_value=("economic_value", "min"),
        mean_economic_value=("economic_value", "mean"),
        maximum_economic_value=("economic_value", "max"),
        number_of_trajectories=("economic_value", "size"),
    )
)

display(comparison_summary)

In [ ]:
detailed_cash_flow_summary = (
    economic_evaluations["detailed_cash_flow"]
    .summary_by_policy
)

economic_component_columns = [
    "stand_id",
    "policy",
    "discounted_gross_revenue",
    "discounted_total_cost",
    "economic_value",
]

available_component_columns = [
    column
    for column in economic_component_columns
    if column in detailed_cash_flow_summary.columns
]

display(
    detailed_cash_flow_summary[
        available_component_columns
    ].head(10)
)

In [ ]:
main_economic_evaluation = tm.evaluate_forest_economics(
    forest,
    economic_mode="detailed_cash_flow",
    discount_rate=DISCOUNT_RATE,
    economic_parameters=DETAILED_CASH_FLOW_PARAMETERS,
)

main_policy_economics = (
    main_economic_evaluation.summary_by_policy.copy()
)

print("Main economic mode: detailed_cash_flow")
print("Currency: constant 2025 USD")
print(f"Real annual discount rate: {DISCOUNT_RATE:.1%}")

display(
    main_policy_economics[
        [
            column
            for column in [
                "stand_id",
                "policy",
                "discounted_gross_revenue",
                "discounted_terminal_value",
                "discounted_total_income",
                "discounted_total_cost",
                "economic_value",
            ]
            if column in main_policy_economics.columns
        ]
    ].head(10)
)

In [ ]:
sample_key = (
    str(example_trajectory.iloc[0]["stand_id"]),
    str(example_trajectory.iloc[0]["policy"]),
)
mode_comparison = []
for mode, evaluation in economic_evaluations.items():
    row = evaluation.summary_by_policy.loc[
        evaluation.summary_by_policy["stand_id"].eq(sample_key[0])
        & evaluation.summary_by_policy["policy"].eq(sample_key[1])
    ].iloc[0]
    mode_comparison.append(
        {
            "mode": mode,
            "metric": evaluation.economic_metric,
            "discounted_harvest_revenue": row["discounted_gross_revenue"],
            "discounted_terminal_value": row["discounted_terminal_value"],
            "discounted_total_income": row["discounted_total_income"],
            "discounted_total_cost": row["discounted_total_cost"],
            "economic_value": row["economic_value"],
        }
    )
display(pd.DataFrame(mode_comparison))

In [ ]:
detailed_cash_flow = economic_evaluations["detailed_cash_flow"].cash_flow_table
display(
    detailed_cash_flow.loc[
        detailed_cash_flow["stand_id"].eq(sample_key[0])
        & detailed_cash_flow["policy"].eq(sample_key[1]),
        [
            "period",
            "economic_event",
            "harvested_dry_wood_t",
            "discounted_gross_revenue",
            "discounted_variable_cost",
            "discounted_transport_cost",
            "discounted_operation_cost_per_t_component",
            "discounted_operation_cost_per_ha_component",
            "discounted_replanting_cost",
            "discounted_annual_management_cost",
            "discounted_terminal_value",
            "discounted_economic_value",
        ],
    ]
)

## 10. Solve one economic optimum under each interpretation

In [ ]:
COMMON_MODEL_OPTIONS = {
    "policy_summary": policy_summary,
    "ending_dry_wood_by_policy": ending_stock,
    "harvested_dry_wood_by_period": harvested_by_period,
    "horizon": HORIZON,
    "forest": forest,
    "carbon_stock_time_by_period": carbon_by_period,
    "discount_rate": DISCOUNT_RATE,
    "minimum_ending_dry_wood_t": 0.0,
    "minimum_ending_dry_wood_fraction": None,
    "even_flow_mode": "none",
}

ECONOMIC_PARAMETERS_BY_MODE = {
    "gross_revenue": GROSS_REVENUE_PARAMETERS,
    "net_unit_value": NET_UNIT_VALUE_PARAMETERS,
    "detailed_cash_flow": BASE_DETAILED_CASH_FLOW_PARAMETERS,
}

In [ ]:
economic_mode_solutions = {}
records = []
for mode, parameters in ECONOMIC_PARAMETERS_BY_MODE.items():
    model = tm.build_forest_management_model(
        **COMMON_MODEL_OPTIONS,
        economic_mode=mode,
        economic_parameters=parameters,
        objective="economic",
    )
    results = tm.solve_model(model, SOLVER_NAME, **SOLVER_OPTIONS)
    solution = tm.extract_solution(model, results)
    economic_mode_solutions[mode] = solution
    records.append(
        {
            "economic_mode": mode,
            "economic_metric": solution["economic_metric"],
            "economic_value": solution["economic_value"],
            "carbon_stock_time_tC_year": solution["carbon_stock_time_tC_year"],
            "ending_dry_wood_t": solution["ending_dry_wood_t"],
            "model_build_time_seconds": solution["model_build_time_seconds"],
            "solve_time_seconds": solution["solve_time_seconds"],
            "total_model_time_seconds": solution["total_model_time_seconds"],
        }
    )
economic_mode_results = pd.DataFrame(records)
display(economic_mode_results)

## 11. Terminal-condition sensitivity

The model can address the end-of-horizon effect in two complementary ways:

1. require a minimum ending dry-wood stock; or
2. add the discounted residual value of standing dry wood at year 30.

The following three models use all 105 stands and differ only in their terminal treatment.

In [ ]:
initial_period_states = forest_long.loc[forest_long["period"].eq(1)]
initial_stock_by_stand = initial_period_states.groupby("stand_id")[
    "standing_dry_wood_t_before_operation"
].agg(["first", "nunique"])
assert initial_stock_by_stand["nunique"].eq(1).all()
INITIAL_DRY_WOOD_TOTAL_T = float(initial_stock_by_stand["first"].sum())
print("Policy-invariant initial landscape stock:", f"{INITIAL_DRY_WOOD_TOTAL_T:,.3f} t")

TERMINAL_SCENARIOS = {
    "no_terminal_treatment": {
        "economic_parameters": BASE_DETAILED_CASH_FLOW_PARAMETERS,
        "minimum_ending_dry_wood_fraction": None,
    },
    "minimum_50_percent_initial_stock": {
        "economic_parameters": BASE_DETAILED_CASH_FLOW_PARAMETERS,
        "minimum_ending_dry_wood_fraction": MINIMUM_ENDING_STOCK_FRACTION,
    },
    "residual_terminal_value": {
        "economic_parameters": DETAILED_CASH_FLOW_PARAMETERS,
        "minimum_ending_dry_wood_fraction": None,
    },
}

terminal_solutions = {}
terminal_records = []
for scenario, settings in TERMINAL_SCENARIOS.items():
    options = dict(COMMON_MODEL_OPTIONS)
    options["minimum_ending_dry_wood_fraction"] = settings[
        "minimum_ending_dry_wood_fraction"
    ]
    model = tm.build_forest_management_model(
        **options,
        economic_mode="detailed_cash_flow",
        economic_parameters=settings["economic_parameters"],
        objective="economic",
    )
    results = tm.solve_model(model, SOLVER_NAME, **SOLVER_OPTIONS)
    solution = tm.extract_solution(model, results)
    terminal_solutions[scenario] = solution
    terminal_records.append(
        {
            "terminal_scenario": scenario,
            "net_present_value": solution["net_present_value"],
            "carbon_stock_time_tC_year": solution["carbon_stock_time_tC_year"],
            "ending_dry_wood_t": solution["ending_dry_wood_t"],
            "initial_dry_wood_total_t": INITIAL_DRY_WOOD_TOTAL_T,
            "ending_to_initial_fraction": (
                solution["ending_dry_wood_t"] / INITIAL_DRY_WOOD_TOTAL_T
            ),
            "model_build_time_seconds": solution["model_build_time_seconds"],
            "solve_time_seconds": solution["solve_time_seconds"],
            "total_model_time_seconds": solution["total_model_time_seconds"],
        }
    )
terminal_results = pd.DataFrame(terminal_records)
display(terminal_results)


## 12. NPV–carbon bi-objective model with residual terminal value

In [ ]:
MODEL_OPTIONS = {
    **COMMON_MODEL_OPTIONS,
    "economic_mode": "detailed_cash_flow",
    "economic_parameters": DETAILED_CASH_FLOW_PARAMETERS,
}

payoff_table = tm.build_biobjective_payoff_table(
    solver_name=SOLVER_NAME,
    solver_options=SOLVER_OPTIONS,
    model_options=MODEL_OPTIONS,
)
display(
    payoff_table[
        [
            "optimized_for",
            "economic_metric",
            "economic_value",
            "carbon_stock_time_tC_year",
            "ending_dry_wood_t",
            "model_build_time_seconds",
            "solve_time_seconds",
            "total_model_time_seconds",
        ]
    ]
)

In [ ]:
economic_anchor_policies = payoff_table.loc[
    payoff_table["optimized_for"].eq("economic"), "selected_policies"
].iloc[0]
selected_keys = set(
    zip(
        economic_anchor_policies["stand_id"].astype(str),
        economic_anchor_policies["policy"].astype(str),
    )
)
selected_long = forest_long.loc[
    forest_long.apply(
        lambda row: (str(row["stand_id"]), str(row["policy"])) in selected_keys,
        axis=1,
    )
].copy()
selected_carbon_by_species = (
    selected_long.groupby("species", as_index=False)
    .agg(
        selected_stands=("stand_id", "nunique"),
        carbon_stock_time_tC_year=("carbon_stock_time_tC_year", "sum"),
        ending_carbon_tC=("standing_wood_carbon_tC_after_operation", lambda s: s[selected_long.loc[s.index, "period"].eq(HORIZON)].sum()),
    )
)
display(selected_carbon_by_species)
print("These totals describe one feasible selected landscape plan.")

In [ ]:
normalization = tm.biobjective_normalization_from_payoff(payoff_table)
print("Payoff-range normalization:")
for key, value in normalization.items():
    print(f"  {key:20s}: {value:,.3f}")

### 12.1 Weighted Pareto front

In [ ]:
weights = np.linspace(0.0, 1.0, WEIGHTED_FRONT_POINTS)
weighted_front = tm.build_weighted_pareto_front(
    weights=weights,
    solver_name=SOLVER_NAME,
    solver_options=SOLVER_OPTIONS,
    model_options=MODEL_OPTIONS,
    payoff_table=payoff_table,
    normalization="payoff_range",
)
display(
    weighted_front[
        [
            "npv_weight",
            "carbon_weight",
            "economic_value",
            "carbon_stock_time_tC_year",
            "ending_dry_wood_t",
            "solve_time_seconds",
            "total_model_time_seconds",
        ]
    ]
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(
    weighted_front["carbon_stock_time_tC_year"],
    weighted_front["economic_value"],
    marker="o",
)
for row in weighted_front.itertuples(index=False):
    ax.annotate(f"w={row.npv_weight:.2f}", (row.carbon_stock_time_tC_year, row.economic_value))
ax.set_xlabel("Standing-wood carbon stock-time (tC·year)")
ax.set_ylabel("Net present value")
ax.set_title("Weighted NPV–carbon solutions")
ax.grid(alpha=0.3)
plt.show()

### 12.2 Epsilon-constraint Pareto front

In [ ]:
epsilon_front = tm.build_epsilon_constraint_front(
    epsilon_objective="carbon",
    number_of_points=EPSILON_FRONT_POINTS,
    payoff_table=payoff_table,
    solver_name=SOLVER_NAME,
    solver_options=SOLVER_OPTIONS,
    model_options=MODEL_OPTIONS,
)
display(
    epsilon_front[
        [
            "epsilon_value",
            "economic_value",
            "carbon_stock_time_tC_year",
            "ending_dry_wood_t",
            "solve_time_seconds",
            "total_model_time_seconds",
        ]
    ]
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(
    epsilon_front["carbon_stock_time_tC_year"],
    epsilon_front["economic_value"],
    marker="o",
    label="Epsilon constraint",
)
ax.scatter(
    weighted_front["carbon_stock_time_tC_year"],
    weighted_front["economic_value"],
    marker="x",
    label="Weighted sum",
)
ax.set_xlabel("Standing-wood carbon stock-time (tC·year)")
ax.set_ylabel("Net present value")
ax.set_title("Bi-objective Pareto approximations — all 105 stands")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 13. Even-flow feasibility on the complete landscape

In [ ]:
even_flow_records = []
for tolerance in (0.25, 0.40, 0.60, 0.80, 1.00):
    even_flow_options = dict(MODEL_OPTIONS)
    even_flow_options.update(
        even_flow_mode="two_sided_average",
        even_flow_tolerance=tolerance,
    )
    model = tm.build_forest_management_model(
        **even_flow_options,
        objective="economic",
    )
    try:
        results = tm.solve_model(model, SOLVER_NAME, **SOLVER_OPTIONS)
        termination = str(results.solver.termination_condition)
        if termination.lower() in {"optimal", "feasible", "locallyoptimal"}:
            solution = tm.extract_solution(model, results)
            even_flow_records.append(
                {
                    "tolerance": tolerance,
                    "termination": termination,
                    "economic_value": solution["economic_value"],
                    "harvest_cv": solution["harvest_schedule"]["harvested_dry_wood_t"].std()
                    / max(solution["harvest_schedule"]["harvested_dry_wood_t"].mean(), 1e-9),
                    "model_build_time_seconds": solution["model_build_time_seconds"],
                    "solve_time_seconds": solution["solve_time_seconds"],
                    "total_model_time_seconds": solution["total_model_time_seconds"],
                    "solution": solution,
                }
            )
        else:
            even_flow_records.append({
                "tolerance": tolerance,
                "termination": termination,
                "model_build_time_seconds": model._treemun_metadata.get("model_build_time_seconds", np.nan),
                "solve_time_seconds": sum(item["solve_time_seconds"] for item in model._treemun_solve_history),
            })
    except RuntimeError as exc:
        even_flow_records.append({"tolerance": tolerance, "termination": str(exc)})

even_flow_results = pd.DataFrame(even_flow_records)
display(even_flow_results.drop(columns=["solution"], errors="ignore"))

## 14. Build full-landscape adjacency

In [ ]:
adjacency_edges = tm.build_adjacency_edges(GEOMETRY_FILE)
final_harvest_indicator = tm.build_final_harvest_indicator(forest)
print("Adjacency edges:", len(adjacency_edges))
print("Final-harvest indicator entries:", len(final_harvest_indicator))
display(adjacency_edges.head())
assert len(adjacency_edges) == 215

In [ ]:
centroids = landscape.set_index("stand_id").geometry.centroid
fig, ax = plt.subplots(figsize=(10, 8))
landscape.plot(ax=ax, facecolor="none", edgecolor="0.55", linewidth=0.4)
for edge in adjacency_edges.itertuples(index=False):
    a = centroids.loc[str(edge.stand_id_a)]
    b = centroids.loc[str(edge.stand_id_b)]
    ax.plot([a.x, b.x], [a.y, b.y], linewidth=0.3, alpha=0.35)
ax.set_title("Adjacency graph — 215 shared-boundary edges")
ax.set_axis_off()
plt.show()

## 15. Green-up event conflicts of the economic optimum

In [ ]:
economic_anchor_solution = payoff_table.loc[
    payoff_table["optimized_for"].eq("economic"), "selected_policies"
].iloc[0]
economic_greenup_events = tm.count_greenup_event_conflicts(
    selected_policies=economic_anchor_solution,
    forest=forest,
    adjacency_edges=adjacency_edges,
    greenup_window=GREENUP_WINDOW,
)
print("Realized green-up event conflicts:", len(economic_greenup_events))
display(economic_greenup_events.head(20))

## 16. Hard green-up model with graceful infeasibility handling

In [ ]:
hard_greenup_model = tm.build_forest_management_model(
    **MODEL_OPTIONS,
    objective="economic",
)
tm.add_greenup_adjacency(
    hard_greenup_model,
    adjacency_edges=adjacency_edges,
    final_harvest_indicator=final_harvest_indicator,
    greenup_window=GREENUP_WINDOW,
    mode="hard",
)

hard_greenup_solution = None
hard_greenup_results = tm.solve_model(
    hard_greenup_model,
    SOLVER_NAME,
    **SOLVER_OPTIONS,
)
hard_termination = str(hard_greenup_results.solver.termination_condition)
print("Hard green-up termination:", hard_termination)
print(
    "Hard-model solver time (s):",
    sum(item["solve_time_seconds"] for item in hard_greenup_model._treemun_solve_history),
)
if hard_termination.lower() in {"optimal", "feasible", "locallyoptimal"}:
    hard_greenup_solution = tm.extract_solution(
        hard_greenup_model,
        hard_greenup_results,
    )
else:
    print("A zero-conflict plan is infeasible for these policies and horizon.")

## 17. Three-objective payoff table: NPV, carbon, and green-up events

In [ ]:
tri_payoff_started = time.perf_counter()
three_objective_payoff = tm.build_three_objective_payoff_table(
    adjacency_edges=adjacency_edges,
    final_harvest_indicator=final_harvest_indicator,
    greenup_window=GREENUP_WINDOW,
    biobjective_payoff_table=payoff_table,
    solver_name=SOLVER_NAME,
    solver_options=SOLVER_OPTIONS,
    model_options=MODEL_OPTIONS,
)
tri_payoff_elapsed = time.perf_counter() - tri_payoff_started
print(f"Three-objective payoff elapsed time: {tri_payoff_elapsed:,.2f} s")
print("Bi-objective anchors reused:", three_objective_payoff.attrs.get("biobjective_anchors_reused"))
display(
    three_objective_payoff[
        [
            "optimized_for",
            "economic_value",
            "carbon_stock_time_tC_year",
            "greenup_event_conflict_count",
            "ending_dry_wood_t",
            "anchor_reused",
            "number_of_solver_calls",
            "solve_time_seconds",
            "total_model_time_seconds",
        ]
    ]
)

## 18. Reused-anchor three-objective epsilon front — all 105 stands

The grid requests six NPV-maximization combinations. Before constructing a new spatial MILP, Treemün checks whether the economic anchor, the lexicographic minimum-green-up anchor, or an already solved looser epsilon model is still feasible and therefore provably optimal for the current combination. Reused points require no additional model construction or solver call.


In [ ]:
tri_front_started = time.perf_counter()
three_objective_front = tm.build_three_objective_epsilon_front(
    adjacency_edges=adjacency_edges,
    final_harvest_indicator=final_harvest_indicator,
    greenup_window=GREENUP_WINDOW,
    number_of_carbon_points=TRI_CARBON_POINTS,
    number_of_greenup_points=TRI_GREENUP_POINTS,
    payoff_table=three_objective_payoff,
    solver_name=SOLVER_NAME,
    solver_options=SOLVER_OPTIONS,
    model_options=MODEL_OPTIONS,
    filter_nondominated=True,
    reuse_payoff_anchors=True,
    reuse_cached_solutions=True,
)
tri_front_elapsed = time.perf_counter() - tri_front_started

print(
    "Requested epsilon combinations:",
    three_objective_front.attrs["requested_combination_count"],
)
print(
    "New spatial MILP solves:",
    three_objective_front.attrs["solved_model_count"],
)
print(
    "Reused optimal solutions:",
    three_objective_front.attrs["reused_model_count"],
)
print(
    "Avoided solver calls:",
    three_objective_front.attrs["avoided_solver_calls"],
)
print(
    "Infeasible combinations:",
    three_objective_front.attrs["infeasible_model_count"],
)
print("Nondominated three-objective solutions:", len(three_objective_front))
print(f"Three-objective front elapsed time: {tri_front_elapsed:,.2f} s")

display(
    three_objective_front[
        [
            "carbon_epsilon_tC_year",
            "greenup_event_conflict_epsilon",
            "solution_source",
            "economic_value",
            "carbon_stock_time_tC_year",
            "greenup_event_conflict_count",
            "ending_dry_wood_t",
            "number_of_solver_calls",
            "solve_time_seconds",
            "total_model_time_seconds",
        ]
    ]
)


In [ ]:
anchor_points = three_objective_payoff.copy()
anchor_points["carbon_epsilon_tC_year"] = np.nan
anchor_points["greenup_event_conflict_epsilon"] = np.nan

three_objective_solution_set = pd.concat(
    [three_objective_front, anchor_points],
    ignore_index=True,
    sort=False,
)
three_objective_solution_set = tm.filter_nondominated_points(
    three_objective_solution_set,
    maximize_columns=("economic_value", "carbon_stock_time_tC_year"),
    minimize_columns=("greenup_event_conflict_count",),
)

knee_analysis = tm.identify_three_objective_knee_point(
    three_objective_solution_set,
    payoff_table=three_objective_payoff,
)
knee_row = knee_analysis.loc[knee_analysis["is_knee_point"]].iloc[0]
knee_solution = knee_row["selected_policies"]

plot_points = knee_analysis.drop_duplicates(
    ["carbon_stock_time_tC_year", "greenup_event_conflict_count"],
    keep="first",
).reset_index(drop=True)

figure = go.Figure()

# Piecewise-planar surface passing through the nondominated points.
if len(plot_points) >= 3:
    xy = plot_points[["normalized_carbon_score", "normalized_greenup_score"]].to_numpy()
    try:
        triangulation = Delaunay(xy)
        figure.add_trace(
            go.Mesh3d(
                x=plot_points["carbon_stock_time_tC_year"],
                y=plot_points["greenup_event_conflict_count"],
                z=plot_points["economic_value"],
                i=triangulation.simplices[:, 0],
                j=triangulation.simplices[:, 1],
                k=triangulation.simplices[:, 2],
                name="Triangulated Pareto surface",
                opacity=0.28,
                color="lightskyblue",
                hoverinfo="skip",
            )
        )
    except QhullError:
        print("The available points are too collinear for Delaunay triangulation.")

# Hyperplane through the three single-objective anchors in normalized space.
plane_point = knee_analysis.attrs.get("anchor_plane_point")
plane_normal = knee_analysis.attrs.get("anchor_plane_normal")
bounds = knee_analysis.attrs["normalization_bounds"]
if plane_point is not None and plane_normal is not None and abs(plane_normal[0]) > 1e-10:
    carbon_min, carbon_max = bounds["carbon_stock_time_tC_year"]
    greenup_min, greenup_max = bounds["greenup_event_conflict_count"]
    economic_min, economic_max = bounds["economic_value"]
    carbon_grid = np.linspace(carbon_min, carbon_max, 16)
    greenup_grid = np.linspace(greenup_min, greenup_max, 16)
    carbon_mesh, greenup_mesh = np.meshgrid(carbon_grid, greenup_grid)
    carbon_score = (carbon_mesh - carbon_min) / max(carbon_max - carbon_min, 1e-12)
    greenup_score = (greenup_max - greenup_mesh) / max(greenup_max - greenup_min, 1e-12)
    economic_score = (
        plane_point[0]
        - plane_normal[1] * (carbon_score - plane_point[1]) / plane_normal[0]
        - plane_normal[2] * (greenup_score - plane_point[2]) / plane_normal[0]
    )
    economic_mesh = economic_min + economic_score * (economic_max - economic_min)
    figure.add_trace(
        go.Surface(
            x=carbon_mesh,
            y=greenup_mesh,
            z=economic_mesh,
            name="Anchor hyperplane",
            opacity=0.15,
            showscale=False,
            colorscale=[[0, "lightgray"], [1, "lightgray"]],
            hoverinfo="skip",
        )
    )

figure.add_trace(
    go.Scatter3d(
        x=plot_points["carbon_stock_time_tC_year"],
        y=plot_points["greenup_event_conflict_count"],
        z=plot_points["economic_value"],
        mode="markers",
        name="Nondominated solutions",
        marker=dict(
            size=7,
            color=plot_points["economic_value"],
            colorscale="Viridis",
            colorbar=dict(title="NPV"),
            line=dict(width=0.5, color="black"),
        ),
        customdata=np.column_stack([
            plot_points["ending_dry_wood_t"],
            plot_points["solve_time_seconds"].fillna(0.0),
        ]),
        hovertemplate=(
            "Carbon: %{x:,.0f} tC·year<br>"
            "Green-up events: %{y:,.0f}<br>"
            "NPV: %{z:,.0f}<br>"
            "Ending dry wood: %{customdata[0]:,.0f} t<br>"
            "Solver time: %{customdata[1]:,.2f} s<extra></extra>"
        ),
    )
)
figure.add_trace(
    go.Scatter3d(
        x=[knee_row["carbon_stock_time_tC_year"]],
        y=[knee_row["greenup_event_conflict_count"]],
        z=[knee_row["economic_value"]],
        mode="markers+text",
        name="Knee point",
        text=["Knee point"],
        textposition="top center",
        marker=dict(size=12, color="red", symbol="diamond", line=dict(width=1, color="black")),
        hovertemplate=(
            "Knee point<br>Carbon: %{x:,.0f} tC·year<br>"
            "Green-up events: %{y:,.0f}<br>NPV: %{z:,.0f}<extra></extra>"
        ),
    )
)

figure.update_layout(
    title="Interactive 3D epsilon-constraint solution set with knee point",
    width=950,
    height=750,
    margin=dict(l=0, r=0, t=70, b=0),
    legend=dict(x=0.02, y=0.98),
    scene=dict(
        xaxis_title="Carbon stock-time (tC·year)",
        yaxis_title="Green-up event conflicts",
        zaxis_title="Net present value",
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.1)),
    ),
)
figure.show()

interactive_html = OUTPUT_DIR / "interactive_three_objective_front_with_knee.html"
figure.write_html(interactive_html, include_plotlyjs=True)
print("Knee method:", knee_analysis.attrs["knee_method"])
print("Interactive HTML:", interactive_html)

## 19. Hyperplane-based knee-point solution

In [ ]:
display(
    knee_analysis.loc[
        knee_analysis["is_knee_point"],
        [
            "economic_value",
            "carbon_stock_time_tC_year",
            "greenup_event_conflict_count",
            "ending_dry_wood_t",
            "knee_score",
            "distance_to_normalized_ideal",
        ],
    ]
)

In [ ]:
selected_keys = set(
    zip(
        knee_solution["stand_id"].astype(str),
        knee_solution["policy"].astype(str),
    )
)
terminal_evaluation = tm.evaluate_forest_economics(
    forest,
    economic_mode="detailed_cash_flow",
    discount_rate=DISCOUNT_RATE,
    economic_parameters=DETAILED_CASH_FLOW_PARAMETERS,
)
knee_economic_summary = terminal_evaluation.summary_by_policy.loc[
    terminal_evaluation.summary_by_policy.apply(
        lambda row: (str(row["stand_id"]), str(row["policy"])) in selected_keys,
        axis=1,
    )
]
display(
    knee_economic_summary[
        [
            "discounted_gross_revenue",
            "discounted_terminal_value",
            "discounted_total_income",
            "discounted_total_cost",
            "economic_value",
        ]
    ].sum().to_frame("landscape_total")
)

## 20. Mathematical-model execution-time summary

In [ ]:
timing_summary = pd.DataFrame(
    [
        {
            "analysis": "Economic-mode optima (3 models)",
            "solver_calls": 3,
            "model_build_time_seconds": economic_mode_results["model_build_time_seconds"].sum(),
            "solve_time_seconds": economic_mode_results["solve_time_seconds"].sum(),
            "total_model_time_seconds": economic_mode_results["total_model_time_seconds"].sum(),
        },
        {
            "analysis": "Terminal sensitivity (3 models)",
            "solver_calls": 3,
            "model_build_time_seconds": terminal_results["model_build_time_seconds"].sum(),
            "solve_time_seconds": terminal_results["solve_time_seconds"].sum(),
            "total_model_time_seconds": terminal_results["total_model_time_seconds"].sum(),
        },
        {
            "analysis": "Bi-objective payoff anchors",
            "solver_calls": payoff_table["number_of_solver_calls"].sum(),
            "model_build_time_seconds": payoff_table["model_build_time_seconds"].sum(),
            "solve_time_seconds": payoff_table["solve_time_seconds"].sum(),
            "total_model_time_seconds": payoff_table["total_model_time_seconds"].sum(),
        },
        {
            "analysis": "Weighted front",
            "solver_calls": weighted_front["number_of_solver_calls"].sum(),
            "model_build_time_seconds": weighted_front["model_build_time_seconds"].sum(),
            "solve_time_seconds": weighted_front["solve_time_seconds"].sum(),
            "total_model_time_seconds": weighted_front["total_model_time_seconds"].sum(),
        },
        {
            "analysis": "Bi-objective epsilon front",
            "solver_calls": epsilon_front["number_of_solver_calls"].sum(),
            "model_build_time_seconds": epsilon_front["model_build_time_seconds"].sum(),
            "solve_time_seconds": epsilon_front["solve_time_seconds"].sum(),
            "total_model_time_seconds": epsilon_front["total_model_time_seconds"].sum(),
        },
        {
            "analysis": "Three-objective payoff (2 new spatial solves)",
            "solver_calls": three_objective_payoff.loc[~three_objective_payoff["anchor_reused"], "number_of_solver_calls"].sum(),
            "model_build_time_seconds": three_objective_payoff.loc[~three_objective_payoff["anchor_reused"], "model_build_time_seconds"].sum(),
            "solve_time_seconds": three_objective_payoff.loc[~three_objective_payoff["anchor_reused"], "solve_time_seconds"].sum(),
            "total_model_time_seconds": three_objective_payoff.loc[~three_objective_payoff["anchor_reused"], "total_model_time_seconds"].sum(),
        },
        {
            "analysis": "Three-objective epsilon front (new solves only)",
            "solver_calls": three_objective_front.attrs["solved_model_count"],
            "model_build_time_seconds": three_objective_front.attrs["model_build_time_seconds"],
            "solve_time_seconds": three_objective_front.attrs["solve_time_seconds"],
            "total_model_time_seconds": three_objective_front.attrs["total_model_time_seconds"],
        },
    ]
)
display(timing_summary)
print("Total recorded solver time (s):", timing_summary["solve_time_seconds"].sum())
print(
    "Three-objective solver calls avoided:",
    three_objective_front.attrs["avoided_solver_calls"],
)


## 21. Export the knee-point solution

In [ ]:
export_solution = {"selected_policies": knee_solution}
output_file = OUTPUT_DIR / "treemun_105_stands_knee_solution.gpkg"
if output_file.exists():
    output_file.unlink()
exported = tm.export_optimal_solution_to_geopackage(
    forest=forest,
    solution=export_solution,
    geometry_file=GEOMETRY_FILE,
    output_file=output_file,
    layer="knee_solution",
)
print("Exported rows:", len(exported))
print("GeoPackage:", output_file)

## 22. Run the automated test suite

In [ ]:
import subprocess

test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)
print(test_result.stdout)
if test_result.returncode != 0:
    print(test_result.stderr)
    raise RuntimeError("The automated test suite failed.")

## Interpretation notes

- Pinus stands can only start from a pre-thinning equation with a valid `next_equation_id`; post-thinning equations are internal states reached after simulated thinning.
- If `initial_age > thinning_age`, the complete initial stock is harvested in period 1 and period 2 restarts at age 1 on the pre-thinning curve.
- All policies for the same stand therefore share one policy-invariant initial stock.
- `gross_revenue` is discounted harvest income and is **not** NPV.
- `net_unit_value` is a discounted margin based on a user-supplied net value per dry tonne.
- `detailed_cash_flow` supports a true NPV with explicit discounted costs and an optional discounted terminal value.
- `minimum_ending_dry_wood_fraction` is an optional terminal stock safeguard relative to the initial landscape stock.
- Candidate carbon summaries report distributions per trajectory; landscape totals are calculated only after one policy is selected per stand.
- `greenup_event_conflict_count` counts conflicting adjacent final-harvest event pairs, not only unique stand pairs.
- Economic and carbon anchors are reused in the three-objective payoff table, so only the spatial anchor requires new MILP solves.
- The three-objective epsilon routine also reuses anchor and cached epsilon solutions whenever optimality is mathematically preserved under stricter bounds.
- `solution_source` distinguishes reused anchors, reused cached epsilon solutions, and newly solved MILPs.
- The notebook requests a 3-by-2 grid using all input stands, but the number of actual solver calls can be smaller than six.
- `threads` and `time_limit_seconds` are translated to native CPLEX, HiGHS, and CBC solver options; the notebook caps threads according to common scheduler allocations.
- The triangulated surface is a piecewise-planar visualization through the sampled nondominated points.
- The knee point is a transparent geometric compromise rule, not a universally preferred solution.
- Model build time and solver time are reported separately because spatial model construction can itself be computationally significant.
